# psycopg2 vs SQLAlchemy, Real Tradeoffs

**Module 4: Python Data Analysis**

**Learning Objectives:**
- Explain what SQLAlchemy's abstraction layer sits on top of, and what that layer actually buys you
- Compare `psycopg2` and SQLAlchemy on pooling, portability, error messages, and learning curve, using running code as the evidence
- Recommend one library for a given script scenario and defend the choice in writing

**Skills:** abstraction layer, connection pool, dialect, ORM


> **You are not presenting this notebook to students.** Use it as your own answer key while you fill in a student notebook live, in front of the class.


## Scenario

You are still on the city agency's data team, and the connection code from last class works. Now a second ask lands: a script that runs every ten minutes, all day, against the same `service_requests` table. Last class you proved both libraries return the same DataFrame. Today you find out whether that means the choice between them does not matter, by measuring what each one actually does to the database server.


## Before You Start

A script opens a database connection inside a loop that runs 500 times. The code is correct and it closes every single one. Is anything still wrong?

*Hint: Think about what the database server has to set up each time a brand-new connection arrives, not about what your Python code does.*


## Documentation

- Every Trip Through the Loop Opens a **New Connection**: https://www.postgresql.org/docs/current/functions-info.html

- One `engine`, and the **Same Connection** Comes Back: https://docs.sqlalchemy.org/en/20/core/pooling.html

- The Same Query, a **Different Database**: https://docs.sqlalchemy.org/en/20/core/engines.html


## Every Trip Through the Loop Opens a **New Connection**

*Context: `pg_backend_pid()` is PostgreSQL's own way of reporting which server process is handling your connection right now.*

**Instructor notes:**

> [ACTIVITY] Everyone runs this cell and reads their three numbers out loud or into chat. You want the room to notice, unprompted, that nobody's three numbers match each other. Do not say it first.
> 
> [SCRIPT] Once the numbers are up, name what they mean. Three different process ids means PostgreSQL started three separate server processes for one loop that asked one question three times. The Python code is completely correct. It closes everything. It is still doing three times the setup work it needed to.
> 
> [SCRIPT] Scale it out loud, because three is not scary. A script that runs every ten minutes all day does this a hundred and forty-four times a day. A dashboard doing it per user request does it thousands of times. That is the cost the next cell removes.
> 
> [MISCONCEPTIONS] Some people will read a different number as a bug or as something they typed wrong. It is not. Different is the expected, correct output here, and the point of the cell.
> 
> [MISCONCEPTIONS] Somebody will ask whether `psycopg2` can pool at all. It can, and saying otherwise would be wrong: `psycopg2.pool` ships `SimpleConnectionPool` and `ThreadedConnectionPool` in the library itself. Two honest qualifications to have ready. You have to reach for it and wire it up yourself, while the other library hands you one without being asked. And psycopg2's own documentation calls that module "simple" and pure Python, so treat it as a real option rather than as an equal to a purpose-built pooler.
> 
> [SCOPE] Someone searching mid-class will find that `psycopg2` is no longer the newest generation, so say it first rather than getting caught. Its successor, psycopg version 3, exists and has a much stronger pool. This module teaches `psycopg2` because it is what SQLAlchemy still reaches for by default on a plain `postgresql://` URL, and because it is what the project's own environment installs. Name it in one sentence and move on; migrating between the two is not this lesson.
> 
> [SOURCES] Verified by running this exact code against a real PostgreSQL 16 table holding this module's own NYC 311 sample: three iterations printed three different backend process ids. `pg_backend_pid()` is documented as returning the process id of the server process attached to the current session (postgresql.org/docs/current/functions-info.html). The `psycopg2.pool` classes named above were confirmed present in psycopg2 2.9.12.

*Input:*


In [ ]:
import os
import psycopg2

url = os.environ["DATABASE_URL"]
for i in range(3):
    conn = psycopg2.connect(url)
    cur = conn.cursor()
    cur.execute("SELECT pg_backend_pid()")
    print("server process:", cur.fetchone()[0])
    cur.close()
    conn.close()


## One `engine`, and the **Same Connection** Comes Back

**Instructor notes:**

> [ACTIVITY] Run it, then ask the room to compare this cell's three numbers against the previous cell's three. Wait for someone to say they are all the same before you say it.
> 
> [SCRIPT] Explain what happened without hand-waving. The engine opened one real connection, lent it to the first `with` block, took it back at the end of the block, and lent the same one out again. Nothing was closed and reopened. That is the pool doing its job, and nobody asked it to.
> 
> [SCRIPT] Read the last line together and translate it, because the wording is confusing on purpose. `Pool size: 5` is the ceiling this engine will keep. `Connections in pool: 1` is how many are actually sitting on the rack right now, which is one because one question at a time never needed a second. A negative overflow number just means the pool has not grown past its own size, so read it as "no overflow," not as an error.
> 
> [SCRIPT] Say the honest scope of this win out loud, because it is the difference between a memorized rule and real judgment. This engine saved two connection setups across three queries. On a script that opens once and exits, it saves nothing at all. Pooling pays off with repetition, and repetition is a property of the script, not of the library.
> 
> [MISCONCEPTIONS] People assume the pool is something you configure before it works. On a normal engine you do not: `create_engine` gives a Postgres connection a pooling engine by default, and the default class is `QueuePool`.
> 
> [MISCONCEPTIONS] A few will expect the `with` block to close the connection, because a `with` block closed things last class. It returns the connection to the pool instead of closing it, which is exactly why the same process id comes back.
> 
> [MISCONCEPTIONS] The last one is worth knowing before a Fellow tries this somewhere else and thinks they broke it. The same-number result holds for a pool living inside your own program, which is what today's cell is. Put a separate connection manager in between your code and the database, or let the pool's own recycle timer fire, and the number can legitimately change between queries. That is not a broken demo, it is a different setup.
> 
> [SOURCES] Verified by running this exact code against a real PostgreSQL 16 instance with SQLAlchemy 2.0.52: three iterations printed the identical backend process id, and `engine.pool.status()` reported `Pool size: 5 Connections in pool: 1`. The default pool class for this engine was confirmed to be `QueuePool` by inspecting `type(engine.pool)`. SQLAlchemy's own pooling documentation states that "the Engine returned by the create_engine() function in most cases has a QueuePool integrated, pre-configured with reasonable pooling defaults," and that `QueuePool` is used for every engine other than SQLite with an in-memory database, where the default is `SingletonThreadPool` instead (docs.sqlalchemy.org/en/20/core/pooling.html). The same page documents the default `pool_size` of five, which is the number printed on this slide.

*Input:*


In [ ]:
from sqlalchemy import create_engine, text

engine = create_engine(url)
q = text("SELECT pg_backend_pid()")

for i in range(3):
    with engine.connect() as conn:
        print("server process:", conn.scalar(q))

print(engine.pool.status())


### 🤔 Check for Understanding: Fresh Connections or a Held Pool

*Mode: Would You Rather*

Would you rather open a fresh connection for each query, or hold one pooled connection open all day?

**Instructor notes:**

> [ACTIVITY] Force a pick before any discussion. In person: everyone stands and moves to one side of the room, then two people from each side say why. Online: everyone types "fresh" or "held" in chat at the same time, then unmute one from each side. Say plainly that a spoken or typed answer counts exactly the same as moving, so nobody is shut out by the format. Four minutes total.
> 
> [SCRIPT] There is genuinely no right answer here, and that is the point. Steer toward the real cost on each side. Fresh every time: the server does full setup work over and over, and a busy script can eat through the server's connection ceiling. Held all day: your script is sitting on a slot nobody else can use, even during the twenty-three hours it is idle, and a connection that has been open for hours can quietly go stale.
> 
> [SCRIPT] Land the transferable idea rather than a verdict. The right answer depends on how often the script asks a question and how long it lives. That is a fact about the script, not about the library, which is exactly how the framework at the end of this lesson is built.
> 
> [MISCONCEPTIONS] Most of the room will pick pooled and stop thinking, because the last cell made pooling look free. Push whoever picks it to name one real cost, and do not accept "none."


## The Same Query, a **Different Database**

*Context: `sqlite://` with nothing after it means a throwaway database that lives in memory and disappears when the notebook closes.*

**Instructor notes:**

> [ACTIVITY] Fill in the blank, 45 seconds, before advancing to the answer. In person: someone says the missing string out loud. Online: everyone types it in chat at the same time on a count of three. This is deliberately the one cell with no separate notebook exercise, so the student notebook shows the same single blank the slide does.
> 
> [SCRIPT] After the reveal, count what changed. One string. Not the query, not `pd.read_sql`, not the `to_sql` call, not a single import. And SQLite is not a variation on Postgres, it is a completely different database with a different driver, which SQLAlchemy picked on its own from the URL.
> 
> [SCRIPT] Read the printed line together. The word `sqlite` is the dialect SQLAlchemy chose, and that is the term itself arriving as output rather than as a claim: a dialect is SQLAlchemy's translation for one specific database and its driver. The `True` is `pg_df.equals(lite_df)`, which compares shape, column names, and every column's type, so the two databases really did hand back the same table.
> 
> [SCOPE] State the limit of this honestly and then stop, because this is the easiest slide in the deck to overclaim from. What just moved was the connection code. Your SQL did not become portable. This query is plain enough to run anywhere, but date functions, data types, and plenty of syntax genuinely differ between databases, and no layer makes that go away. Say that sentence out loud rather than leaving it in notes.
> 
> [MISCONCEPTIONS] Someone will conclude they should always use SQLAlchemy "in case we switch databases." Most teams never switch. The real everyday version of this benefit is smaller and more useful: running the same code against a throwaway local database for testing and against the real one in production.
> 
> [MISCONCEPTIONS] A few will expect `psycopg2` to do this too. It cannot, and that is not a flaw. It is a PostgreSQL adapter on purpose, and being one specific database's driver is the whole reason it exists.
> 
> [SOURCES] Verified by running this exact code: the SQLite engine reported `dialect.name` as `sqlite` with driver `pysqlite`, the Postgres engine reported `postgresql` with driver `psycopg2`, and `pg_df.equals(lite_df)` printed `True`. SQLAlchemy's engine documentation describes the database URL as carrying the dialect and driver it selects (docs.sqlalchemy.org/en/20/core/engines.html). The scope caveat above is the important one: SQLAlchemy's own documentation is explicit that a textual SQL string is passed through to the database essentially as written, so dialect differences in the SQL itself remain the author's problem.

*Input:*


In [ ]:
import pandas as pd

sql = "SELECT borough, status FROM service_requests"
with engine.connect() as conn:
    pg_df = pd.read_sql(sql, conn)

lite = create_engine("sqlite://")
pg_df.to_sql("service_requests", lite, index=False)
lite_df = pd.read_sql(sql, lite)

print(lite.dialect.name, pg_df.equals(lite_df))
